# Evaluation of the recommender

Reproduces `scripts/evaluate.py` and looks at the queries that miss. Data: `data/postings.csv` (synthetic, see README) and `data/eval/`.

In [ ]:
import sys
sys.path.insert(0, '../src'); sys.path.insert(0, '../scripts')
import pandas as pd
from jobrec.data import load_postings, load_eval
from jobrec.embed import TfidfEmbedder, GloveEmbedder, VECTORS_PATH
from jobrec.recommend import Recommender
from evaluate import evaluate

postings = load_postings('../data/postings.csv')
queries, relevant = load_eval('../data/eval')
len(postings), len(queries)

## Precision@k and MRR per embedder and α

In [ ]:
rows = []
backends = [('tfidf', TfidfEmbedder)]
if VECTORS_PATH.exists():
    backends.append(('glove', GloveEmbedder))
for name, cls in backends:
    for alpha in (0.0, 0.5, 1.0):
        rec = Recommender(postings, cls(), alpha=alpha)
        res = evaluate(rec, queries, relevant)
        rows.append({'embedder': name, 'alpha': alpha, **res})
pd.DataFrame(rows)

## Which queries miss, and what they get instead

In [ ]:
rec = Recommender(postings, TfidfEmbedder(), alpha=0.5)
for q in queries.itertuples(index=False):
    top = rec.recommend(q.cv_text, 5)
    wrong = [(r.posting.id, r.posting.role_family, r.posting.title) for r in top
             if r.posting.id not in relevant[q.id]]
    if wrong:
        print(q.id, q.role_family, '->', wrong)

## One query in detail: matched and missing skills

In [ ]:
q = queries.iloc[6]  # Q07, the incubator CV
print(q.cv_text)
for r in rec.recommend(q.cv_text, 5):
    print(f'{r.score:.2f} sim={r.similarity:.2f} cov={r.skill_coverage:.2f} '
          f'{r.posting.id} {r.posting.title} | {r.explanation}')